In [ ]:
!pip install scikit-uplift -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from scipy import stats
from sklift.datasets import fetch_hillstrom

In [ ]:
dataset = fetch_hillstrom(
    target_col="all"
)

X = dataset.data.copy()
y = dataset.target.copy()
treatment = dataset.treatment.copy()

hillstrom = X.copy()

hillstrom["segment"] = treatment
hillstrom["visit"] = y["visit"]
hillstrom["conversion"] = y["conversion"]
hillstrom["spend"] = y["spend"]

hillstrom.head()

In [ ]:
hillstrom["segment"].value_counts()

In [ ]:
ab = hillstrom[
    hillstrom["segment"].isin(
        ["No E-Mail", "Mens E-Mail"]
    )
].copy()

In [ ]:
spend_summary = (
    ab.groupby("segment")
      ["spend"]
      .agg(
          ["count", "mean", "median", "std", "sum"]
      )
)

spend_summary

In [ ]:
mean_control = (
    ab.loc[
        ab["segment"] == "No E-Mail",
        "spend"
    ]
    .mean()
)

mean_treatment = (
    ab.loc[
        ab["segment"] == "Mens E-Mail",
        "spend"
    ]
    .mean()
)

revenue_lift = (
    mean_treatment
    - mean_control
)

print(
    f"Control revenue/customer: "
    f"${mean_control:.2f}"
)

print(
    f"Treatment revenue/customer: "
    f"${mean_treatment:.2f}"
)

print(
    f"Incremental revenue/customer: "
    f"${revenue_lift:.2f}"
)

In [ ]:
control_spend = (
    ab.loc[
        ab["segment"] == "No E-Mail",
        "spend"
    ]
    .to_numpy()
)

treatment_spend = (
    ab.loc[
        ab["segment"] == "Mens E-Mail",
        "spend"
    ]
    .to_numpy()
)

In [ ]:
rng = np.random.default_rng(6021)
B = 5000

boot_spend_diff = np.empty(B)

for b in range(B):

    control_star = rng.choice(
        control_spend,
        size=len(control_spend),
        replace=True
    )

    treatment_star = rng.choice(
        treatment_spend,
        size=len(treatment_spend),
        replace=True
    )

    boot_spend_diff[b] = (
        treatment_star.mean()
        - control_star.mean()
    )

In [ ]:
plt.figure(figsize=(9,5))

plt.hist(
    boot_spend_diff,
    bins=40,
    edgecolor="black"
)

plt.axvline(
    0,
    linestyle="--"
)

plt.axvline(
    revenue_lift
)

plt.xlabel(
    "Difference in Mean Spend ($)"
)

plt.ylabel(
    "Bootstrap Frequency"
)

plt.title(
    "Bootstrap Distribution of Incremental Revenue"
)

plt.show()

In [ ]:
lower, upper = np.percentile(
    boot_spend_diff,
    [2.5, 97.5]
)

print(
    f"Observed revenue lift: "
    f"${revenue_lift:.2f}"
)

print(
    f"95% bootstrap CI: "
    f"[${lower:.2f}, ${upper:.2f}]"
)

**A/B Testing Proportions**

In [ ]:
summary = (
    hillstrom
    .groupby("segment")
    .agg(
        customers=("segment", "size"),
        visit_rate=("visit", "mean"),
        conversion_rate=("conversion", "mean"),
        mean_spend=("spend", "mean"),
        total_revenue=("spend", "sum")
    )
)

summary

In [ ]:
visit_summary = (
    ab.groupby("segment")
      ["visit"]
      .agg(["count", "sum", "mean"])
)

visit_summary

In [ ]:
p_control = visit_summary.loc[
    "No E-Mail",
    "mean"
]

p_treatment = visit_summary.loc[
    "Mens E-Mail",
    "mean"
]

visit_lift = (
    p_treatment - p_control
)

print(
    f"Control visit rate: {p_control:.2%}"
)

print(
    f"Treatment visit rate: {p_treatment:.2%}"
)

print(
    f"Absolute lift: {visit_lift:.2%}"
)

In [ ]:
control_visit = (
    ab.loc[
        ab["segment"] == "No E-Mail",
        "visit"
    ]
    .astype(int)
    .to_numpy()
)

treatment_visit = (
    ab.loc[
        ab["segment"] == "Mens E-Mail",
        "visit"
    ]
    .astype(int)
    .to_numpy()
)

In [ ]:
B = 5000

boot_visit_diff = np.empty(B)

for b in range(B):

    control_star = rng.choice(
        control_visit,
        size=len(control_visit),
        replace=True
    )

    treatment_star = rng.choice(
        treatment_visit,
        size=len(treatment_visit),
        replace=True
    )

    boot_visit_diff[b] = (
        treatment_star.mean()
        - control_star.mean()
    )

In [ ]:
plt.figure(figsize=(9,5))

plt.hist(
    boot_visit_diff,
    bins=40,
    edgecolor="black"
)

plt.axvline(
    0,
    linestyle="--"
)

plt.axvline(
    visit_lift
)

plt.xlabel(
    "Difference in Visit Rates"
)

plt.ylabel(
    "Bootstrap Frequency"
)

plt.title(
    "Bootstrap Distribution: Men's E-Mail vs No E-Mail"
)

plt.show()

In [ ]:
lower, upper = np.percentile(
    boot_visit_diff,
    [2.5, 97.5]
)

print(
    f"Observed lift: {visit_lift:.4f}"
)

print(
    f"95% bootstrap CI: "
    f"[{lower:.4f}, {upper:.4f}]"
)

**ANOVA**

In [ ]:
hillstrom.groupby("segment")["spend"].agg(
    ["count", "mean", "std"]
).round(2)

In [ ]:
fig = px.box(
    hillstrom,
    x="segment",
    y="spend",
    color="segment",
    points="outliers",
    category_orders={
        "segment": [
            "No E-Mail",
            "Mens E-Mail",
            "Womens E-Mail"
        ]
    },
    labels={
        "segment": "Email Treatment",
        "spend": "Customer Spending ($)"
    },
    title="Customer Spending Across Email Treatments"
)

fig.update_layout(
    xaxis_title="Email Treatment",
    yaxis_title="Customer Spending ($)",
    showlegend=False
)

fig.show()

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [ ]:
model = ols(
    "spend ~ C(segment)",
    data=hillstrom
).fit()

In [ ]:
anova_table = sm.stats.anova_lm(model, typ=2)

anova_table

In [ ]:
np.random.seed(6021)

groups = [
    "No E-Mail",
    "Mens E-Mail",
    "Womens E-Mail"
]

comparisons = [
    ("Mens E-Mail", "No E-Mail"),
    ("Womens E-Mail", "No E-Mail"),
    ("Mens E-Mail", "Womens E-Mail")
]

B = 5000
alpha = 0.05

alpha_corrected = alpha / len(comparisons)

results = []

for group1, group2 in comparisons:

    x1 = hillstrom.loc[
        hillstrom["segment"] == group1, "spend"
    ].values

    x2 = hillstrom.loc[
        hillstrom["segment"] == group2, "spend"
    ].values


    observed_diff = x1.mean() - x2.mean()

    bootstrap_diffs = []

    for b in range(B):

        sample1 = np.random.choice(
            x1,
            size=len(x1),
            replace=True
        )

        sample2 = np.random.choice(
            x2,
            size=len(x2),
            replace=True
        )

        bootstrap_diffs.append(
            sample1.mean() - sample2.mean()
        )

    lower = np.percentile(
        bootstrap_diffs,
        100 * alpha_corrected / 2
    )

    upper = np.percentile(
        bootstrap_diffs,
        100 * (1 - alpha_corrected / 2)
    )

    results.append({
        "comparison": f"{group1} - {group2}",
        "difference": observed_diff,
        "lower": lower,
        "upper": upper
    })

results = pd.DataFrame(results)

results.round(3)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=results["difference"],
        y=results["comparison"],
        mode="markers",
        marker=dict(size=10),
        error_x=dict(
            type="data",
            symmetric=False,
            array=results["upper"] - results["difference"],
            arrayminus=results["difference"] - results["lower"]
        ),
        name="Mean Difference"
    )
)


fig.add_vline(
    x=0,
    line_dash="dash",
    line_width=2
)

fig.update_layout(
    title="Bootstrap Confidence Intervals for Pairwise Mean Differences",
    xaxis_title="Difference in Mean Spending ($)",
    yaxis_title="",
    showlegend=False,
    height=450
)

fig.show()